# 00 · Code-along: from the NeoWs docs to a feature

**NASA Space Apps Cairo 2026 · Bootcamp · API & Data Integration**

**Goal:** a list of the *closest asteroids this week*, built from a real NASA API call, with a saved backup and one handled error.

**Done means:** one real call returns `200 OK` · the fields you need are in a clean table with units · the table is saved as JSON/CSV · one feature uses it · at least one error is handled with a clear message.

> ✍️ **Starter notebook.** Cells marked `TODO` are yours to finish. Green ✅ check cells tell you when a step works.

Open in Colab: *File → Save a copy in Drive* first, so your edits are kept.

## Setup
Run the next cell. Get your own free key at <https://api.nasa.gov> first and paste it in.

In [ ]:
# ▶ Run this cell first
import os, json, requests
import pandas as pd
from datetime import date, timedelta

API_KEY = "DEMO_KEY"   # 👉 replace with YOUR free key from https://api.nasa.gov (DEMO_KEY runs out fast in a full room)
# Never paste a paid key (for example an AI key) into a notebook you share.

def check(test, good, hint):
    """Tiny self-check: prints ✅ when the step works, ❌ plus a hint when it does not."""
    try:
        ok = bool(test())
    except Exception:
        ok = False
    print(("✅ " + good) if ok else ("❌ " + hint))

def show_url(url, params):
    """Print a request URL with your key hidden, so it is safe to show on a projector."""
    req = requests.Request("GET", url, params=params).prepare()
    key = str(params.get("api_key", ""))
    print(req.url.replace(key, "YOUR_KEY") if key else req.url)

In [ ]:
# 🛟 Safety net: an ILLUSTRATIVE sample in the same shape as a real NeoWs answer.
# These are NOT real asteroids. It only exists so your feature can still show something if the API is down.
SAMPLE_NEOWS = {'element_count': 5,
 'near_earth_objects': {'2025-10-01': [{'id': '90001',
                                        'name': 'SAMPLE-1 (illustrative)',
                                        'is_potentially_hazardous_asteroid': False,
                                        'estimated_diameter': {'meters': {'estimated_diameter_min': 54.2,
                                                                          'estimated_diameter_max': 120.5}},
                                        'close_approach_data': [{'close_approach_date': '2025-10-01',
                                                                 'relative_velocity': {'kilometers_per_second': '12.3'},
                                                                 'miss_distance': {'kilometers': '5041879.1'}}]},
                                       {'id': '90002',
                                        'name': 'SAMPLE-2 (illustrative)',
                                        'is_potentially_hazardous_asteroid': False,
                                        'estimated_diameter': {'meters': {'estimated_diameter_min': 20.3,
                                                                          'estimated_diameter_max': 45.2}},
                                        'close_approach_data': [{'close_approach_date': '2025-10-01',
                                                                 'relative_velocity': {'kilometers_per_second': '8.7'},
                                                                 'miss_distance': {'kilometers': '11250430.8'}}]}],
                        '2025-10-02': [{'id': '90003',
                                        'name': 'SAMPLE-3 (illustrative)',
                                        'is_potentially_hazardous_asteroid': True,
                                        'estimated_diameter': {'meters': {'estimated_diameter_min': 139.5,
                                                                          'estimated_diameter_max': 310.0}},
                                        'close_approach_data': [{'close_approach_date': '2025-10-02',
                                                                 'relative_velocity': {'kilometers_per_second': '19.4'},
                                                                 'miss_distance': {'kilometers': '2310560.2'}}]},
                                       {'id': '90004',
                                        'name': 'SAMPLE-4 (illustrative)',
                                        'is_potentially_hazardous_asteroid': False,
                                        'estimated_diameter': {'meters': {'estimated_diameter_min': 10.3,
                                                                          'estimated_diameter_max': 22.8}},
                                        'close_approach_data': [{'close_approach_date': '2025-10-02',
                                                                 'relative_velocity': {'kilometers_per_second': '6.1'},
                                                                 'miss_distance': {'kilometers': '31870210.5'}}]},
                                       {'id': '90005',
                                        'name': 'SAMPLE-5 (illustrative)',
                                        'is_potentially_hazardous_asteroid': False,
                                        'estimated_diameter': {'meters': {'estimated_diameter_min': 39.7,
                                                                          'estimated_diameter_max': 88.3}},
                                        'close_approach_data': [{'close_approach_date': '2025-10-02',
                                                                 'relative_velocity': {'kilometers_per_second': '14.0'},
                                                                 'miss_distance': {'kilometers': '7640120.9'}}]}]}}

def sample_banner():
    print("⚠️  USING ILLUSTRATIVE SAMPLE DATA, not real NASA data. Fix the call to see real results.")

## Steps 01–03 · Need, find the API, read the docs
Open the **NeoWs** documentation on <https://api.nasa.gov> (section *Asteroids NeoWs*) and find the **Feed** endpoint.

Fill in what the docs tell you. Look for: base URL, endpoint, parameters, limits.

### Your turn: fill in the plan

In [ ]:
NEED = "..."          # one sentence: what will your feature show?
API_CHOICE = "..."    # which NASA API fits that need?
BASE_URL = "..."      # from the docs
ENDPOINT = "..."      # the path of the Feed endpoint
MAX_DAYS = ...        # the longest date range one call may ask for

In [ ]:
check(lambda: BASE_URL.startswith("https://api.nasa.gov") and ENDPOINT.endswith("/feed") and MAX_DAYS == 7,
      "Plan matches the docs.",
      "Re-read the NeoWs Feed section: base URL, endpoint path and the maximum date range.")

## Step 04 · Test the URL in the browser first
Build the parameters, print the URL, then **paste it into a browser tab** (use your real key). Firefox shows JSON nicely. Seeing the answer before coding saves time.

### Your turn: parameters

In [ ]:
URL = BASE_URL + ENDPOINT
END = date.today()
START = ...            # TODO: 6 days before END (the range may not be longer than 7 days)

params = {
    "start_date": ...,   # TODO: dates as YYYY-MM-DD text (hint: START.isoformat())
    "end_date": ...,
    "api_key": API_KEY,
}
show_url(URL, params)

In [ ]:
check(lambda: len(params["start_date"]) == 10 and (END - START).days <= 7,
      "Parameters look right. Now open the printed URL in a browser tab.",
      "start_date and end_date must be YYYY-MM-DD text, at most 7 days apart.")

## Step 05 · Code the call
Same request, now from Python. `timeout` stops a slow API from freezing your notebook.

### Your turn: make the call

In [ ]:
resp = requests.get(..., params=..., timeout=15)   # TODO: fill in URL and params
print("Status:", resp.status_code)

In [ ]:
check(lambda: resp.status_code == 200,
      "200 OK. Your call works!",
      f"Status was {getattr(resp, 'status_code', '?')}. 403 = key, 429 = too many calls, 400 = bad parameter (see the status-code slide).")

## Step 06 · Inspect the JSON
Look before you parse. What is at the top level? How many objects? What does ONE asteroid look like?

### Your turn: explore

In [ ]:
feed = resp.json()
# TODO: print the top-level keys
# TODO: print element_count and the list of dates in "near_earth_objects"
# TODO: print ONE asteroid nicely: json.dumps(one_asteroid, indent=2)

In [ ]:
check(lambda: "near_earth_objects" in feed, "You found the data inside the response.",
      "Look for the key that holds the asteroids, grouped by date.")

## Step 07 · Pick the five fields
Keep only: **name, size, speed, miss distance (km), hazardous flag**. Walk the boxes: dates → asteroids → first close approach.

<details><summary>Hint</summary>

`estimated_diameter` → `meters` → `estimated_diameter_max` · `close_approach_data[0]` → `relative_velocity` → `kilometers_per_second` · `close_approach_data[0]` → `miss_distance` → `kilometers` · `is_potentially_hazardous_asteroid`

</details>

### Your turn: flatten the JSON into rows

In [ ]:
rows = []
for day, asteroids in feed["near_earth_objects"].items():
    for a in asteroids:
        ca = ...                      # TODO: the first close approach of this asteroid
        rows.append({
            "date": day,
            "name": ...,
            "diameter_m": ...,        # estimated_diameter -> meters -> estimated_diameter_max
            "speed_kms": ...,         # relative_velocity -> kilometers_per_second
            "miss_km": ...,           # miss_distance -> kilometers
            "hazardous": ...,         # is_potentially_hazardous_asteroid
        })
print(len(rows), "asteroids")

In [ ]:
check(lambda: len(rows) == feed["element_count"], "Every asteroid is in your rows.",
      "Count mismatch: loop over every date AND every asteroid in each date.")

## Step 08 · Clean: types, units, order
JSON gave numbers **as text** (`"5041879.1"`). Convert them, then add a friendlier unit: the Moon is about 384,400 km away, so `miss_moons = miss_km / 384_400` says "how many Moon-distances".

### Your turn: make a clean table

In [ ]:
df = pd.DataFrame(rows)
print(df.dtypes)
# TODO: convert diameter_m, speed_kms and miss_km to float
# TODO: add miss_moons = miss_km / 384_400, rounded to 2 decimals
# TODO: sort by miss_km (smallest first) and reset the index
df.head()

In [ ]:
check(lambda: df["miss_km"].dtype.kind == "f" and df["miss_km"].is_monotonic_increasing and "miss_moons" in df.columns,
      "Clean table: numbers are numbers, sorted, with a friendly unit.",
      "miss_km must be float, sorted ascending, and you need a miss_moons column.")

## Step 09 · Save a backup the moment it works
⚠️ NeoWs copies **your key** into the `links` it returns. Redact the key before saving, so it never leaks in a file you share.

### Your turn: save JSON and CSV

In [ ]:
os.makedirs("backup", exist_ok=True)
# TODO: turn feed into text, replace API_KEY with "REDACTED", turn it back into a dict
safe_feed = ...
with open("backup/neows_feed.json", "w") as f:
    json.dump(safe_feed, f)
# TODO: save the clean table as backup/neows_clean.csv (no index column)
print(os.listdir("backup"))

In [ ]:
check(lambda: os.path.exists("backup/neows_feed.json") and os.path.exists("backup/neows_clean.csv")
      and API_KEY not in open("backup/neows_feed.json").read() or API_KEY == "DEMO_KEY",
      "Backup saved, key hidden.", "Save both files, and make sure your key is not inside the JSON.")

## Step 10 · Build the feature
Show the closest 10 asteroids, with the friendly unit, plus a chart.

### Your turn: the feature

In [ ]:
closest = ...      # TODO: the 10 closest asteroids; columns: name, date, miss_km, miss_moons, diameter_m, hazardous
display(closest)
# TODO: a horizontal bar chart of miss_moons for those 10 (hint: closest.plot.barh(...))

In [ ]:
check(lambda: len(closest) <= 10 and closest["miss_km"].is_monotonic_increasing,
      "Feature on screen.", "Show at most 10 rows, closest first.")

## Step 10b · Wrap your steps into a function
We will need the same parsing twice (live data and the saved copy).

### Your turn: `feed_to_df`

In [ ]:
def feed_to_df(feed):
    """Turn a NeoWs feed (dict) into the clean table from steps 07 and 08."""
    # TODO: paste your steps 07 and 08 here and return the DataFrame
    ...

In [ ]:
check(lambda: len(feed_to_df(feed)) == len(df), "Same table, now reusable.", "The function should return the same table as before.")

## Step 11 · Handle errors: a clear message, then your saved copy
Real demos break: wrong key (403), too many calls (429), an empty answer, a slow server. Your function should **never crash and never show a blank screen**.

Rules for `get_feed`:
1. Call with a `timeout`.
2. Turn 401/403 and 429 into a clear message.
3. Treat an empty result as a failure too.
4. On any failure: print the message, then load `backup/neows_feed.json`; if that does not exist, use the sample (with the warning banner).

### Your turn: `get_feed`

In [ ]:
def get_feed(start, end, key):
    """Return (feed_dict, source) where source is 'live', 'saved copy' or 'sample'."""
    try:
        r = requests.get(URL, params={"start_date": start, "end_date": end, "api_key": key}, timeout=15)
        # TODO: raise ValueError with a CLEAR message for 401/403 and for 429
        # TODO: r.raise_for_status() for other errors, then data = r.json()
        # TODO: raise ValueError("Empty result") if there are no near_earth_objects
        # TODO: return data, "live"
        ...
    except (requests.RequestException, ValueError) as e:
        print("⚠️  API problem:", e)
        # TODO: load backup/neows_feed.json if it exists and return (data, "saved copy")
        # TODO: otherwise sample_banner() and return (SAMPLE_NEOWS, "sample")
        ...

### 🎬 The planned failure
Run with `USE_WRONG_KEY = True` and watch the clear message and the fallback. Then flip it to `False` and run again.

In [ ]:
USE_WRONG_KEY = True    # planned failure: set to False when you are done

feed, source = get_feed(START.isoformat(), END.isoformat(), "WRONG_KEY" if USE_WRONG_KEY else API_KEY)
print("Data source:", source)
df = feed_to_df(feed)
display(df.head(10)[["name", "date", "miss_km", "miss_moons", "diameter_m", "hazardous"]])

## ✅ You are done when
- [ ] one real call returned `200 OK` with your own key
- [ ] the fields are in a clean table with units
- [ ] `backup/` holds a JSON and a CSV file (key hidden)
- [ ] one feature is on screen
- [ ] one error was handled with a clear message

**Next:** pick an API card (`01`, `02` or `03`). Fast pairs: try `04` to join a second source.